# Compare ablation methods

**Role:** Analysis.

Evaluate marker zeroing, trained masking and context-matched fate replacement on saved models, then compare their total and size-dependent curvature responses. The default selection is the completed `Ndependent_20260922_115100` masking run: depth-2 FiLM models with all five validation folds and seed 42. Zeroing and replacement use the 0% masking control; masking uses the 5% checkpoint. This compares both intervention definitions and training conditions, since the masking model differs from the control. No model or baseline is retrained.

All methods share coverage sampling, fold splits, exact neighbor hops and supplied cell counts. Curvature, predictions and anatomical labels never enter sampling or donor matching. Replacement uses only training organoids, includes unassigned as an identity, and by default freezes replacement weights at each source organoid's observed N during sweeps. We intersect supported physical source cases before plotting, average within organoids and give organoids equal weight. Sample-count heatmaps precede the total-effect heatmaps and observed/sweep curves. Outputs and settings are saved under the selected training run, with resumable per-checkpoint shards. Previously completed ablation folders can also be compared without new inference.


In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/models/gnn.py").is_file())  # Locate the repository from the notebook’s working directory.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ROOT = PROJECT_ROOT  # Repository root used to resolve data and result paths.

## Analysis settings

Select the training run, model depth and method/checkpoint rates here. Every saved fold at that depth is included; `MODEL_SEEDS=None` includes every saved seed. The 0% and 5% models must use the same folds, seeds and input bundles. To isolate intervention definitions on a single trained model instead, give every method the same positive masking rate.

Shared coverage settings match the individual ablation notebooks. `RUN_INFERENCE=True` evaluates missing checkpoint shards; `False` loads a completed analysis with the same configuration. Alternatively set `ANALYSIS_DIRS` to a dictionary of labels and completed output folders to skip model loading and inference. `SWEEP_COUNTS=[]` produces only the total comparison.

The notebook is configured to evaluate the 5% masking checkpoints when run. `ANALYSIS_DIRS=None` uses the model selection above, and `RUN_INFERENCE=True` computes the comparison. Results are saved under the new `masking_0_vs_5pct` analysis tag.


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from src.analysis.interventions.replacement import MatchConfig

# Saved model selection
TRAINING_SUBFOLDER = 'fate_masking_training'  # Workflow folder under training_results/.
TRAINING_RUN = 'Ndependent_20260922_115100'  # Completed masking run, including its 0% control.
MODEL_DEPTH = 2  # Analyze every exact hop from 1 to this depth.
MODEL_NAME = 'film'  # Saved model family: gin or film.
HIDDEN_DIM = 128  # Saved hidden width at the selected depth.
MODEL_SEEDS = None  # None includes every saved seed; every saved fold is always included.

# Intervention and checkpoint training rates
METHODS_TO_COMPARE = {
    'Marker zeroing (0%)': {'method': 'marker_zeroing', 'rate': 0.0},  # Use the unmasked control.
    'Masking (5%)': {'method': 'masking', 'rate': 0.05},  # Hide fate using the 5% masking-trained model.
    'Replacement (0%)': {'method': 'replacement', 'rate': 0.0},  # Use the same control as marker zeroing.
}
ANALYSIS_DIRS = None  # Optional {label: completed analysis folder}; None evaluates the models selected above.

# Coverage sampling (restored from 9ae3657: ablation_analysis)
SAMPLING_SCHEME = 'coverage'  # coverage: pair/hop minimum targets; alternatives: uniform or stratified.
CENTER_SAMPLE_SIZE = 2000  # Maximum recipient centers per validation fold; None keeps all (coverage only).
CENTER_MIN_MARKER_COUNT = 50  # Target centers per recipient identity before non-overlap filtering.
CENTER_MIN_PAIR_COUNT = 25  # Target cases per recipient/source identity at EACH hop before filtering.
SAMPLING_SEED = 0  # Original SUBGRAPH_SEED; fold index is added, identical across model seeds/methods.

# Source-cell independence
CENTER_APPLY_NON_OVERLAP = True  # At hop d, retained sources block others within graph distance d.
CENTER_NON_OVERLAP_GROUP_COLUMNS = ('hop', 'center_marker', 'source_marker')  # Apply independently within each pair/hop.

# Alternative sampling schemes
CENTERS_PER_ORGANOID = 5  # Only uniform/stratified: recipients per organoid (per identity for stratified).

# Replacement reference (ignored by other methods)
N_DEPENDENT_REPLACEMENT = False  # False freezes weights at observed N; True recomputes weights at swept N.
MATCHING = MatchConfig(
    neighbors=256,  # Candidate donor cells for local matching.
    max_size_ratio=2.,  # Maximum donor/source count ratio.
    max_degree_ratio=2.,  # Maximum donor/source degree ratio.
    max_composition_l1=1.,  # Maximum neighborhood-composition mismatch.
    min_cells=20,  # Minimum matched donor cells.
    min_organoids=5,  # Minimum donor organoids.
    min_identity_cells=5,  # Minimum cells for each replacement fate.
    min_identity_organoids=3,  # Minimum organoids for each replacement fate.
    max_donors_per_organoid=256,  # Donor quota per training organoid.
)

# Effect scale and uncertainty
GEOMETRIC_NORMALIZATION = False  # Also compute ΔK/(4π/A_ref(N)); A_ref is fitted to training organoids only.
PLOT_METRIC = 'delta_mu'  # delta_mu (curvature), delta_z (transformed), or delta_relative (requires normalization).
BOOTSTRAP_SAMPLES = 500  # Organoid bootstrap resamples for uncertainty intervals.
SUMMARY_SEED = 42  # Bootstrap seed shared across methods.
HEATMAP_DISPLAY_MIN_CASES = 20  # Minimum retained unique cases; int or hop dict, e.g. {1:20,2:30}; also gates size curves.
MIN_PLOT_ORGANOIDS = 1  # Minimum organoid support per displayed pair.

# Execution and output
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'  # Inference device.
BATCH_SIZE = 128  # Maximum edited subgraphs per prediction batch.
RUN_INFERENCE = True  # False reloads this exact completed analysis without inference.
ANALYSIS_TAG = 'masking_0_vs_5pct'  # Output label; changed inference settings select a separate version.
OUTPUT_DIR = None  # None saves below this training run; optional comparison-figure directory override.

# Size dependence
SWEEP_COUNTS = [80,150,300,550,700]  # Fixed-graph supplied N values; [] skips sweeps.
SIZE_BINS = [0,150,250,350,500,800,np.inf]  # Bins for effects at observed organoid sizes.


## Restore paired checkpoints and validation inputs

Resolve all selected folds and seeds for each training rate, and check that their saved input bundles agree. Only the 5% model receives the unknown-fate flag during masking; the 0% control receives ordinary observed inputs during zeroing and replacement. Baseline offsets, target transforms and global scaling come from the saved run. Changing inference settings or code selects a separate output version instead of mixing cached results.


In [ ]:
from dataclasses import asdict
from src.artifacts.paths import resolve_training_run
from src.artifacts.runs import AnalysisRun, select_depth_records
from src.artifacts.provenance import workflow_fingerprint, analysis_output_path
from src.analysis.interventions.masking import checked_settings
from src.analysis.interventions.fate_edits import (
    METHODS, fate_graphs, sample_fate_contexts, build_replacement_reference,
    replacement_distribution, evaluate_fate_edit, add_geometric_scale,
)

# An explicit folder mapping keeps the previous load-only comparison available.
USE_SAVED_OUTPUTS = ANALYSIS_DIRS is not None
if not USE_SAVED_OUTPUTS:
    if MODEL_DEPTH < 1 or len(METHODS_TO_COMPARE) < 2:
        raise ValueError('Choose at least two methods and a model depth >= 1.')
    if Path(ANALYSIS_TAG).name != ANALYSIS_TAG or ANALYSIS_TAG in ('.', '..', ''):
        raise ValueError('ANALYSIS_TAG must be a simple folder name.')
    if PLOT_METRIC == 'delta_relative' and not GEOMETRIC_NORMALIZATION:
        raise ValueError('Enable geometric normalization to plot delta_relative.')
    counts = list(SWEEP_COUNTS)
    if len(set(counts)) != len(counts) or any(not np.isfinite(n) or n <= 0 for n in counts):
        raise ValueError('Sweep counts must be distinct, positive and finite.')
    RUN_DIR = resolve_training_run(ROOT, TRAINING_SUBFOLDER, TRAINING_RUN)
    run = AnalysisRun(RUN_DIR, root=ROOT)
    method_records = {}
    for label, spec in METHODS_TO_COMPARE.items():
        if spec['method'] not in METHODS:
            raise ValueError(f"Unknown intervention: {spec['method']}")
        if spec['method'] == 'masking' and (spec['rate'] is None or spec['rate'] <= 0):
            raise ValueError('Masking requires a positive training rate.')
        method_records[label] = select_depth_records(run, MODEL_DEPTH, model_name=MODEL_NAME,
            hidden_dim=HIDDEN_DIM, seeds=MODEL_SEEDS, rate=spec['rate'])
    # Pair folds/seeds before inference, not merely after pooling predictions.
    first_records = next(iter(method_records.values()))
    paired_keys = ['fold', 'seed']
    if 'input_bundle' in first_records:
        paired_keys.append('input_bundle')
    expected_inputs = first_records[paired_keys].reset_index(drop=True)
    for label, records in method_records.items():
        if not records[paired_keys].reset_index(drop=True).equals(expected_inputs):
            raise ValueError(f'{label}: checkpoints must use identical folds, seeds and saved inputs.')
    display(pd.DataFrame([dict(method=label, family=records.iloc[0]['name'], depth=MODEL_DEPTH,
        width=int(records.iloc[0].hidden_dim), masking_rate=METHODS_TO_COMPARE[label]['rate'],
        folds=records.fold.nunique(), seeds=sorted(records.seed.unique()), checkpoints=len(records))
        for label, records in method_records.items()]))
    print('Training run:', RUN_DIR)
    print('Validation inputs are restored from each checkpoint; no training is performed.')
    width = int(first_records.iloc[0].hidden_dim)
    comparison_base = RUN_DIR/'analysis'/'ablation_comparison'/f'{first_records.iloc[0]["name"]}_d{MODEL_DEPTH}_h{width}'/ANALYSIS_TAG
    code_sha256 = workflow_fingerprint(PROJECT_ROOT/'experiments/ablation/ablation_comparison.ipynb')
    configs_to_run = {}
    for label, records in method_records.items():
        spec = METHODS_TO_COMPARE[label]
        configs_to_run[label] = dict(training_run=str(RUN_DIR.resolve()), model_keys=records.key.tolist(),
            method=spec['method'], depth=MODEL_DEPTH, sampling_scheme=SAMPLING_SCHEME,
            centers=CENTERS_PER_ORGANOID, sampling_seed=SAMPLING_SEED,
            center_sample_size=CENTER_SAMPLE_SIZE, center_min_marker_count=CENTER_MIN_MARKER_COUNT,
            center_min_pair_count=CENTER_MIN_PAIR_COUNT, center_apply_non_overlap=CENTER_APPLY_NON_OVERLAP,
            center_non_overlap_group_columns=list(CENTER_NON_OVERLAP_GROUP_COLUMNS),
            n_dependent_replacement=N_DEPENDENT_REPLACEMENT if spec['method'] == 'replacement' else False,
            matching=asdict(MATCHING), counts=counts, geometric_normalization=GEOMETRIC_NORMALIZATION,
            view='size' if counts else 'total', code_sha256=code_sha256)
    comparison_config = dict(analyses=configs_to_run)
    comparison_dir = analysis_output_path(comparison_base, comparison_config)
    if OUTPUT_DIR is None:
        OUTPUT_DIR = comparison_dir
    if RUN_INFERENCE:
        checked_settings(comparison_dir/'settings.json', comparison_config)
    ANALYSIS_DIRS = {}
    for index, (label, spec) in enumerate(METHODS_TO_COMPARE.items()):
        ANALYSIS_DIRS[label] = comparison_dir/f"{index}_{spec['method']}_p{spec['rate']:g}"
    print('Comparison outputs:', OUTPUT_DIR)
else:
    if OUTPUT_DIR is None:
        OUTPUT_DIR = Path(next(iter(ANALYSIS_DIRS.values()))).parent/'comparison'
OUTPUT_DIR = Path(OUTPUT_DIR)


## Evaluate the three interventions

For every fold, reuse the same deterministic coverage sampling at exact hops 1 through the model depth. The recipient and graph remain unchanged. Replacement enumerates alternative identities from training-only context matches; fixed weights are estimated at observed N and held unchanged throughout a sweep. An all-zero source is a real unassigned identity: zeroing it is a no-op, while masking hides that identity and replacement substitutes another fate.

Evaluate observed size and each supplied N on these fixed cases. Save support audits, exact organoid membership and case-level outputs for each checkpoint. Completed checkpoint shards are reused on repeated runs with identical settings. The full comparison may take time, particularly for replacement, which evaluates multiple possible fates per source.


In [ ]:
if not USE_SAVED_OUTPUTS:
    for label, records in method_records.items():
        spec = METHODS_TO_COMPARE[label]
        config = configs_to_run[label]
        method_dir = Path(ANALYSIS_DIRS[label])
        if RUN_INFERENCE:
            method_dir.mkdir(parents=True,exist_ok=True)
            checked_settings(method_dir/'settings.json',config)
            (method_dir/'cases').mkdir(exist_ok=True)
            (method_dir/'support').mkdir(exist_ok=True)
            for record in records.itertuples():
                path = method_dir/'cases'/f'{record.key}.csv.gz'
                if path.exists():
                    print(f'{label}: reuse {record.key}')
                    continue
                selection = run.select(record.key,device=DEVICE)
                fold_seed = SAMPLING_SEED + int(record.fold)
                subgraphs,manifest,sampling_info = sample_fate_contexts(fate_graphs(selection),selection['marker_names'],MODEL_DEPTH,
                    scheme=SAMPLING_SCHEME,centers=CENTERS_PER_ORGANOID,seed=fold_seed,
                    max_subgraphs=CENTER_SAMPLE_SIZE,min_center_count=CENTER_MIN_MARKER_COUNT,
                    min_pair_count=CENTER_MIN_PAIR_COUNT,apply_non_overlap=CENTER_APPLY_NON_OVERLAP,
                    non_overlap_group_columns=CENTER_NON_OVERLAP_GROUP_COLUMNS,return_info=True)
                for table_name in ('center_coverage','pair_coverage','case_non_overlap'):
                    sampling_info[table_name].to_csv(method_dir/'support'/f'{record.key}_{table_name}.csv.gz',index=False)
                print(f"{label} — sampling {record.key}: {sampling_info['n_selected']} recipient centers before case filtering")
                if manifest.empty:
                    raise ValueError(f'No eligible neighboring source cells for {record.key}.')
                marker_names = selection['marker_names']
                observed_weights = None
                if spec['method'] == 'replacement':
                    matcher,contexts = build_replacement_reference(selection,manifest,MODEL_DEPTH,config=MATCHING,seed=fold_seed)
                    observed_weights,donor_audit = replacement_distribution(manifest,contexts,matcher,marker_names)
                    donor_audit.to_csv(method_dir/'support'/f'{record.key}_observed.csv.gz',index=False)
                pieces = []
                for count in [None,*counts]:
                    weights = observed_weights
                    if spec['method'] == 'replacement' and count is not None and N_DEPENDENT_REPLACEMENT:
                        weights,donor_audit = replacement_distribution(manifest,contexts,matcher,marker_names,count=count)
                        donor_audit.to_csv(method_dir/'support'/f'{record.key}_N{count:g}.csv.gz',index=False)
                    frame = evaluate_fate_edit(selection,subgraphs,manifest,spec['method'],count=count,weights=weights,
                        device=DEVICE,batch_size=BATCH_SIZE)
                    frame['fold'],frame['seed'],frame['model_key'] = record.fold,record.seed,record.key
                    if GEOMETRIC_NORMALIZATION:
                        frame,area_reference = add_geometric_scale(frame,selection)
                        (method_dir/'support'/f'{record.key}_area_reference.json').write_text(json.dumps(area_reference,indent=2))
                    pieces.append(frame)
                saved = pd.concat(pieces,ignore_index=True)
                saved['center_marker_names'] = saved.center_marker_names.map(json.dumps)
                temporary = path.with_suffix('.tmp')
                saved.to_csv(temporary,index=False,compression='gzip')
                temporary.replace(path)
                membership = {role:[g.organoid_str for g in selection['groups'][role]] for role in ('train','val')}
                (method_dir/'support'/f'{record.key}_membership.json').write_text(json.dumps(membership,indent=2))
                print(f'{record.key}: {len(manifest)} fixed source cases, {len(counts)} supplied counts')
                del selection,subgraphs
            (method_dir/'complete.json').write_text(json.dumps({'checkpoints':len(records)}))


## Validate inputs and align source cases

Restrict to unchanged recipient/source identities and identical graph/fate inputs. Require support at every supplied N for sweep comparisons. The settings check also applies when loading externally supplied analysis folders. Intersect cases before averaging so differences in donor support cannot silently change the compared populations. Counts may therefore be lower than the sampling targets, especially for rare replacement identities.


In [ ]:
from src.analysis.interventions.fate_edits import load_ablation_result, align_method_cases, summarize_fate_effects
from src.plotting.influence_maps import plot_pair_heatmaps, plot_pair_size_curves
if len(ANALYSIS_DIRS)<2 or any(path is None for path in ANALYSIS_DIRS.values()):
    raise ValueError('Set each ANALYSIS_DIRS entry to a completed ablation output directory (or remove unused entries).')
tables,configs = {},{}
for label,path in ANALYSIS_DIRS.items():
    tables[label],configs[label] = load_ablation_result(Path(path))
    if not USE_SAVED_OUTPUTS and configs[label] != configs_to_run[label]:
        raise ValueError(f'{label}: saved inference settings differ from this configuration.')
    if PLOT_METRIC not in tables[label]:
        raise ValueError(f'{label} does not contain {PLOT_METRIC}.')
if len({cfg['depth'] for cfg in configs.values()})!=1:
    raise ValueError('Compare results at the same model depth.')
count_grids = {tuple(cfg['counts']) for cfg in configs.values()}
if len(count_grids)>1:
    raise ValueError('Use the same sweep counts in every result (or all total-only results).')
sampling_keys = ['sampling_scheme','centers','sampling_seed','center_sample_size',
    'center_min_marker_count','center_min_pair_count','center_apply_non_overlap','center_non_overlap_group_columns','sampling_options']
if any('center_sample_size' not in cfg for cfg in configs.values()):
    raise ValueError('These results predate restored coverage sampling. Rerun ablation inference with matching settings; no retraining is needed.')
sampling_settings = pd.DataFrame({label:{key:cfg.get(key) for key in sampling_keys} for label,cfg in configs.items()}).T
if len({json.dumps({key:cfg.get(key) for key in sampling_keys},sort_keys=True) for cfg in configs.values()}) != 1:
    raise ValueError('Use matching sampling settings across methods; inspect the saved sampling settings.')
PLOT_IDENTITIES = list(dict.fromkeys(name for path,cfg in zip(ANALYSIS_DIRS.values(),configs.values())
    for name in pd.read_csv(Path(path)/'support'/f"{cfg['model_keys'][0]}_pair_coverage.csv.gz").center_marker.unique()))
checkpoint_sets = {(cfg['training_run'],tuple(cfg['model_keys'])) for cfg in configs.values()}
print('Same trained checkpoints: method comparison.' if len(checkpoint_sets)==1 else
      'Different checkpoints: effects combine ablation-method and model-training differences.')
aligned = align_method_cases(tables)
support = pd.DataFrame([dict(label=label, sampled_cases=len(table), common_rows=len(aligned[label]),
    common_organoids=aligned[label].organoid_str.nunique()) for label,table in tables.items()])
print(f"Common validation organoids: {support.common_organoids.iloc[0]}; detailed support saved to common_support.csv.")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
(OUTPUT_DIR/'sources.json').write_text(json.dumps(configs,indent=2))
support.to_csv(OUTPUT_DIR/'common_support.csv',index=False)

sampling_settings.to_csv(OUTPUT_DIR/'sampling_settings.csv',index_label='label')


## Shared sample-count heatmaps

All methods use the same physical cases after intersecting donor support. Show these counts once, with a panel for each hop, rather than repeating them for every method. We verify that the retained counts agree across methods. If observed and sweep cohorts contain exactly the same cases, one figure covers both; otherwise show a separate shared figure for each cohort (possible with N-dependent replacement).

Rows are center identities and columns are source identities. Every cell contains its unique case count on a logarithmic blue color scale. Model seeds and repeated N evaluations do not add samples. Low-support pairs remain visible here even when their response cells are hatched. Detailed per-method coverage audits remain saved with the inference outputs.


In [ ]:
from src.analysis.interventions.fate_edits import count_fate_cases, CASE_KEYS
from src.plotting.influence_maps import plot_pair_count_heatmaps
shared_frame = next(iter(aligned.values()))
comparison_counts = count_fate_cases(shared_frame,metric=PLOT_METRIC)
count_keys = ['analysis','hop','center_marker','source_marker_name']
shared_counts = comparison_counts.sort_values(count_keys).reset_index(drop=True)
for label,frame in aligned.items():
    method_counts = count_fate_cases(frame,metric=PLOT_METRIC).sort_values(count_keys).reset_index(drop=True)
    pd.testing.assert_frame_equal(shared_counts,method_counts,obj=f'Common counts for {label}')
comparison_counts.to_csv(OUTPUT_DIR/'pair_sample_counts.csv',index=False)
count_limit = max(2,float(comparison_counts.n_cases.max())) if not comparison_counts.empty else 2
count_hops = range(1,next(iter(configs.values()))['depth']+1)
count_modes = ['observed','sweep'] if next(iter(configs.values()))['counts'] else ['observed']
shared_across_sizes = False
if 'sweep' in count_modes:
    observed_cases = pd.MultiIndex.from_frame(shared_frame.loc[shared_frame.analysis=='observed',CASE_KEYS]).unique()
    sweep_cases = pd.MultiIndex.from_frame(shared_frame.loc[shared_frame.analysis=='sweep',CASE_KEYS]).unique()
    shared_across_sizes = observed_cases.difference(sweep_cases).empty and sweep_cases.difference(observed_cases).empty
    if shared_across_sizes:
        count_modes = ['observed']
for mode in count_modes:
    selected = comparison_counts[comparison_counts.analysis==mode]
    description = ('observed and all sweep sizes' if shared_across_sizes else
        'pooled observed sizes' if mode=='observed' else 'fixed all-N sweep cohort')
    fig = plot_pair_count_heatmaps(selected,hops=count_hops,marker_names=PLOT_IDENTITIES,vmax=count_limit,
        title=f'Shared samples, all methods: {description}')
    suffix = 'shared' if shared_across_sizes else mode
    fig.savefig(OUTPUT_DIR/f'pair_sample_counts_{suffix}.png',dpi=160,bbox_inches='tight')
    plt.show()


## Total comparison at observed sizes

For each method separately, average its observed-size ablation effects within each organoid and then across organoids on the common case set. The total is **not** an average of the fixed-graph N-sweep curve. Similar totals can coexist with different N-dependent curves, and positive and negative effects at different observed sizes can cancel in the pooled mean.

Heatmaps share a symmetric color scale across methods. Small differences can look similar on this common scale; the precise means are exported to `total_comparison.csv`. Each panel names its hop; no hop-zero intervention is included. Hatched cells lack sufficient support.

**Reading the scales:** near-white does not mean zero. The heatmap uses one color range for all marker pairs and methods, whereas each size-curve panel chooses its own y-axis limits. Compare the numeric axis scales before interpreting the visual magnitude. Marker zeroing of an unassigned source is a no-op.


In [ ]:
(OUTPUT_DIR/'plot_settings.json').write_text(json.dumps(dict(
    metric=PLOT_METRIC,heatmap_display_min_cases=HEATMAP_DISPLAY_MIN_CASES,
    min_plot_organoids=MIN_PLOT_ORGANOIDS,bootstrap_samples=BOOTSTRAP_SAMPLES),indent=2))
total = pd.concat([summarize_fate_effects(frame,view='total',metric=PLOT_METRIC,
    draws=BOOTSTRAP_SAMPLES,seed=SUMMARY_SEED).assign(label=label) for label,frame in aligned.items()],ignore_index=True)
total.to_csv(OUTPUT_DIR/'total_comparison.csv',index=False)
limit = float(total['mean'].abs().max()) or 1.
for label,part in total.groupby('label',sort=False):
    fig = plot_pair_heatmaps(part,title=f'{label}: {PLOT_METRIC}, pooled observed sizes',
        min_organoids=MIN_PLOT_ORGANOIDS,min_cases=HEATMAP_DISPLAY_MIN_CASES,marker_names=PLOT_IDENTITIES,limit=limit,hops=range(1,next(iter(configs.values()))['depth']+1))
    fig.savefig(OUTPUT_DIR/f'total_{list(aligned).index(label)}.png',dpi=160,bbox_inches='tight')
    plt.show()


## Size-dependent comparison

Overlay methods within each recipient/source pair and exact hop. Solid curves group different observed organoids by N; dashed curves sweep N on fixed graphs. Bands are pointwise organoid-bootstrap intervals conditional on the fitted models.

Observed and sweep curves use distinct colors, with matching uncertainty bands. Observed curves also use solid lines and circles; sweeps use dashed lines and squares. Each figure title identifies the exact hop distance.


In [ ]:
if next(iter(configs.values()))['counts']:
    size = pd.concat([summarize_fate_effects(frame,view='size',metric=PLOT_METRIC,size_bins=SIZE_BINS,
        draws=BOOTSTRAP_SAMPLES,seed=SUMMARY_SEED).assign(label=label) for label,frame in aligned.items()],ignore_index=True)
    size.to_csv(OUTPUT_DIR/'size_comparison.csv',index=False)
    for hop in sorted(size.hop.unique()):
        fig = plot_pair_size_curves(size,hop=hop,title=f'Common source cases: {PLOT_METRIC}',
            min_organoids=MIN_PLOT_ORGANOIDS,min_cases=HEATMAP_DISPLAY_MIN_CASES)
        fig.savefig(OUTPUT_DIR/f'size_hop{hop}.png',dpi=160,bbox_inches='tight')
        plt.show()
else:
    print('Selected total-only results. Select outputs of size_dependent_ablation for size comparisons.')
